# SysMLv2 訳語集 Geminiワークブック

## 初期化

In [ ]:
%run gemini.py

In [ ]:
g_client = GAClient()

In [ ]:
g_client.upload_standards()

## 設定・保守

In [ ]:
g_client.list_files()

In [ ]:
g_client.cleanup_files()

In [ ]:
g_client.set_light_model()

## 抽出

In [ ]:
# --- [一括抽出フェーズ] ---

target_sections = [f"9.{i}" for i in range(1, 9)]

# 実行ループ
for section in target_sections:
    print(f"\n[開始] Section {section} ---")
    g_client.extract_and_save_with_retry(section)
    time.sleep(30) # 成功後も少し余裕を持たせる

print("\n=== すべての抽出が完了しました！ ===")

単体で抽出する場合 (7.2と7.3を抽出)

In [ ]:
g_client.extract_and_save_with_retry(['7.2', '7.3'])

## 統合作業

In [ ]:
import pandas as pd
import glob
import os
import csv

# 1. 統一したい「正しい列名」の定義
# あなたが最終的に手元に残したい順番で定義してください
target_columns = ["Term (EN)", "Equivalent (JA)", "Definition (EN)", "Context/Example (EN)", "Section", "File"]

csv_files = glob.glob("output/SysML2-Section-*.csv")
print(f"--- 統合開始（列名統一モード）: {len(csv_files)} ファイル ---")

dfs = []
for f in csv_files:
    try:
        # 一度普通に読み込む
        df = pd.read_csv(f, encoding="utf-8-sig")
        
        # 💡 ここがポイント：最初の4列だけを抜き出し、名前を強制的に統一する
        # これにより "Term (EN)" と "Term(EN)" の違いを抹殺します
        df_cleaned = df.iloc[:, :6].copy() 
        df_cleaned.columns = target_columns
        
        # 出所がわかるようにセクション名を追加
        section_name = os.path.basename(f).replace("SysML2-Section-", "").replace(".csv", "")
        df_cleaned['Source'] = section_name
        
        dfs.append(df_cleaned)
    except Exception as e:
        print(f"⚠️ スキップ: {f} ({e})")

if not dfs:
    print("ファイルが見つかりませんでした。")
else:
    # 2. 結合（列名が統一されているので、綺麗に縦に並びます）
    df_master = pd.concat(dfs, ignore_index=True)

    # 3. データとして混入したヘッダー行を削除
    df_master = df_master[df_master["Term (EN)"] != "Term (EN)"]
    df_master = df_master[df_master["Term (EN)"] != "Term(EN)"]

    # 4. Term (EN) でソート
    df_master = df_master.sort_values(by="Term (EN)", key=lambda x: x.str.lower())

    # 5. QUOTE_MINIMAL で保存
    output_file = "output/SysML2_Master.csv"
    df_master.to_csv(
        output_file, 
        index=False, 
        encoding="utf-8-sig",
        quoting=csv.QUOTE_MINIMAL
    )

    print(f"\n--- 統合完了 ---")
    print(f"列名を統一して {len(df_master)} 件を保存しました: {output_file}")

## 分析

In [ ]:
# Term (EN) 列のユニークな値の数をカウント
unique_count = df_master["Term (EN)"].nunique()

print(f"--- 統計情報 ---")
print(f"延べ件数（重複含む）: {len(df_master)} 件")
print(f"正味件数（ユニーク）: {unique_count} 件")
print(f"重複している件数     : {len(df_master) - unique_count} 件")

In [ ]:
# Term (EN) は同じだが、Equivalent (JP) が異なる組み合わせを抽出
diff_translation = df_master.groupby("Term (EN)")["Equivalent (JA)"].nunique()
inconsistent_terms = diff_translation[diff_translation > 1].index.tolist()

if inconsistent_terms:
    print(f"⚠️ 翻訳が揺れている用語が {len(inconsistent_terms)} 件あります。")
    print(df_master[df_master["Term (EN)"].isin(inconsistent_terms)].sort_values("Term (EN)"))
else:
    print("✨ 素晴らしい！全ての重複用語で翻訳（Equivalent）が完全に一致しています。")

In [ ]:
# 翻訳が揺れている行だけを抽出して保存
df_inconsistent = df_master[df_master["Term (EN)"].isin(inconsistent_terms)].sort_values("Term (EN)")

# CSVとして保存（QUOTE_MINIMAL を適用）
df_inconsistent.to_csv("SysML2_Translation_Conflicts.csv", index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)

print("ゆらぎリストを 'SysML2_Translation_Conflicts.csv' に保存しました。")